# 02 - Explore Dataset & Preliminary Cleaning

We are now going to look at the merged dataset and get an intuition of what's happening.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dft = pd.read_csv('../data/STATS19/dft_STATS19_1979_23_SY.csv', low_memory=False)

In [2]:
# dft.info()

In [6]:
def nulls_and_minus1_summary(df):
    summary = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Null Count': df.isnull().sum().values,
        '% Missing': (df.isnull().sum() / len(df) * 100).round(2),
        '% -1 Values': [
            ((df[col] == -1) | (df[col] == -1.0)).sum() / len(df) * 100 
            if pd.api.types.is_numeric_dtype(df[col]) else None
            for col in df.columns
        ]
    })
    return summary.sort_values(by='Null Count', ascending=False).reset_index(drop=True)

# Generate and display the table
null_minus1_summary = nulls_and_minus1_summary(dft)
with pd.option_context('display.max_rows', None):
    display(null_minus1_summary)

,Column,Data Type,Null Count,% Missing,% -1 Values
0,dir_to_n,float64,237598,97.70,0.000000
1,dir_to_e,float64,237598,97.70,0.000000
2,dir_from_n,float64,237569,97.69,0.000000
3,dir_from_e,float64,237569,97.69,0.000000
4,latitude,float64,121895,50.12,0.000000
5,longitude,float64,121895,50.12,0.000000
6,location_easting_osgr,float64,3525,1.45,0.000000
7,location_northing_osgr,float64,3525,1.45,0.000000
8,carriageway_hazards,int64,0,0.00,0.057979
9,vehicle_direction_from,int64,0,0.00,0.208889


We can highlight following columns that have some missing values:
- location_easting_osgr

In [ ]:
columns_to_drop = [
  'dir_from_e',
  'dir_from_n',
  'dir_to_e',
  'dir_to_n',
  'latitude',
  'longitude',
  'location_easting_osgr',
  'location_northing_osgr',
  'did_police_officer_attend_scene_of_accident',
  'enhanced_casualty_severity',
  'casualty_distance_banding',
  'driver_distance_banding',
  'pedestrian_road_maintenance_worker',
  'casualty_imd_decile',
  'vehicle_left_hand_drive'
]
dft_filtered = dft.drop(columns=columns_to_drop)
dft_filtered.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243191 entries, 0 to 243190
Data columns (total 70 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   accident_index                           243191 non-null  object 
 1   accident_year                            243191 non-null  int64  
 2   accident_reference                       243191 non-null  object 
 3   vehicle_reference                        243191 non-null  int64  
 4   casualty_reference                       243191 non-null  int64  
 5   casualty_class                           243191 non-null  int64  
 6   sex_of_casualty                          243191 non-null  int64  
 7   age_of_casualty                          243191 non-null  int64  
 8   age_band_of_casualty                     243191 non-null  int64  
 9   casualty_severity                        243191 non-null  int64  
 10  pedestrian_location             

### Missing Value Analysis
Now that we have explored the dataset a bit, we can see some columns have missing data, especially the older data. So, we will do some analysis of the missing values, as follows:
1. Check for null values in all columns.
2. Visualise the missing values by year.

After the analysis, we will replace '-1' values with NaN, so the dataset is more compatible with Pandas.

In [20]:
def plot_missing_values_by_year(df, year_col='accident_year'):
    # Step 1: Select columns with at least one null
    missing_cols = df.columns[df.isnull().any()].tolist()

    # Step 2: Group by year and count missing values for each column
    missing_by_year = df.groupby(year_col)[missing_cols].apply(lambda x: x.isnull().sum())

    # Step 3: Plot each column in a subplot
    num_cols = len(missing_cols)
    cols = 3
    rows = (num_cols + cols - 1) // cols  # auto-determine rows based on cols

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), sharex=True)
    axes = axes.flatten()

    for i, col in enumerate(missing_cols):
        axes[i].plot(missing_by_year.index, missing_by_year[col], marker='o')
        axes[i].set_title(f"{col} - Missing Count")
        axes[i].set_xlabel("Year")
        axes[i].set_ylabel("Missing")
        axes[i].grid(True)

    # Remove any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# 🔧 Call the function on your dataset
# plot_missing_values_by_year(dft)


### Checking missing values for each parameter over the years
We will use an interactive tool that lets us check one parameter at a time and explore how missing values change over the years.

By selecting a column from the dropdown, we can see the percentage of missing values (including both NaN and -1) across different years. This helps us:
* Understand when certain features start being recorded properly
* Spot sudden changes in data quality or reporting
* Decide which years to include in our analysis based on data completeness

In [19]:
import pandas as pd
import numpy as np
%pip install plotly ipywidgets
import plotly.graph_objs as go
from ipywidgets import interact, Dropdown

def get_missing_percent_series(df, year_col, col_name):
    if pd.api.types.is_numeric_dtype(df[col_name]):
        is_missing = df[col_name].isnull() | (df[col_name] == -1) | (df[col_name] == -1.0)
    else:
        is_missing = df[col_name].isnull()

    years = sorted(df[year_col].unique())
    year_counts = df[year_col].value_counts().sort_index()
    yearly_missing = df[is_missing].groupby(df[year_col]).size()
    percent_missing = (yearly_missing / year_counts * 100).reindex(years, fill_value=0)

    return percent_missing

def interactive_missing_plot(df, year_col='accident_year'):
    dropdown = Dropdown(
        options=[col for col in df.columns if col != year_col],
        description='Column:',
        layout={'width': '600px'},
        style={'description_width': 'initial'}
    )

    def plot_column(col_name):
        percent_missing = get_missing_percent_series(df, year_col, col_name)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=percent_missing.index,
            y=percent_missing.values,
            mode='lines+markers',
            name=col_name
        ))
        fig.update_layout(
            title=f"Missing or -1 Values Over the Years: {col_name}",
            xaxis_title="Year",
            yaxis_title="Percent Missing",
            yaxis_range=[0, 100],
            template="plotly_white"
        )
        fig.show()

    interact(plot_column, col_name=dropdown)

# Run it in a notebook
interactive_missing_plot(dft)


^C


interactive(children=(Dropdown(description='Column:', layout=Layout(width='600px'), options=('accident_index',…

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/14.8 MB 10.7 MB/s eta 0:00:02
   ------------ --------------------------- 4.5/14.8 MB 10.7 MB/s eta 0:00:01
   ----------------- ---------------------- 6.6/14.8 MB 10.9 MB/s eta 0:00:01
   ------------------------ --------------- 8.9/14.8 MB 10.9 MB/s eta 0:00:01
   ----------------------------- ---------- 11.0/14.8 MB 10.7 MB/s eta 0:00:01
   ------------------------------------ --- 13.4/14.8 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 14.8/14.8 MB 10.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
